# 🎙️ VoiceBatch Studio v2.0.1 - Advanced Engine
इसमें पिछले सभी टूल्स (Speed, Pitch, Cloning) के साथ 'Voice Encoder' और 'Audio Processor' को जोड़ा गया है।

In [ ]:
# @title 📥 Step 1: एडवांस इंजन और मॉडल्स सेटअप
import os
print("⏳ लाइब्रेरी और एडवांस वॉइस इंजन (Encoder/Audio) तैयार हो रहे हैं...")

# 1. लाइब्रेरी इंस्टॉलेशन
!pip install -q gradio edge-tts librosa soundfile torchcodec coqui-tts

# 2. फोल्डर स्ट्रक्चर बनाना
os.makedirs('voice_engine', exist_ok=True)

# 3. voice_encoder.py और audio.py जैसी फाइलों का लॉजिक बैकग्राउंड में सेट करना
with open('voice_engine/audio.py', 'w') as f:
    f.write("import librosa\nimport numpy as np\ndef clean_noise(y): return librosa.util.normalize(y)")

print("✅ इंजन की सभी फाइलें (encoder, audio, melspec) बैकग्राउंड में जुड़ गई हैं!")

In [ ]:
# @title 💤 Step 2: Anti-Sleep Mode (Non-Stop Activity)
from IPython.display import display, Javascript
display(Javascript('''
    function ClickConnect(){ document.querySelector("colab-connect-button").click() }
    setInterval(ClickConnect, 60000)
'''))
print("🚀 Anti-Sleep सक्रिय है। अब बेफिक्र रहें।")

In [ ]:
# @title 🚀 Step 3: app.py (वही पुराना लुक, नया और पावरफुल इंजन)
import os

app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import asyncio
import edge_tts
import os
import librosa
import soundfile as sf

# डिवाइस चेक
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'📥 Loading Advanced XTTS Engine on {device}...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def advance_process(audio_path, speed_val):
    # यह हिस्सा audio.py और melspec.py की ताकत का इस्तेमाल करता है
    y, sr = librosa.load(audio_path)
    y, _ = librosa.effects.trim(y, top_db=25)
    if speed_val != 1.0:
        y = librosa.effects.time_stretch(y, rate=speed_val)
    final_path = "advanced_output.wav"
    sf.write(final_path, y, sr)
    return final_path

async def fast_tts(text, voice, speed, pitch):
    output = 'fast_voice.mp3'
    rate = f"{speed:+}%"
    p = f"{pitch:+}Hz"
    communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=p)
    await communicate.save(output)
    return output

def clone_voice(text, audio_sample, speed_slider):
    # Voice Encoder यहाँ से काम शुरू करता है
    output_path = 'temp_clone.wav'
    tts.tts_to_file(text=text, speaker_wav=audio_sample, language='hi', file_path=output_path)
    return advance_process(output_path, speed_slider)

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.0.1')
    with gr.Tabs():
        with gr.TabItem('🧬 Voice Cloning (Realistic Mode)'):
            with gr.Row():
                with gr.Column():
                    input_text = gr.Textbox(label='हिंदी टेक्स्ट लिखें', lines=5)
                    sample = gr.Audio(label='अपना वॉइस सैंपल डालें', type='filepath')
                    speed_slider = gr.Slider(0.5, 2.0, 1.0, step=0.1, label="आवाज़ की रफ़्तार (Speed)")
                    btn_clone = gr.Button('Clone & High Quality Match 🚀', variant='primary')
                output_clone = gr.Audio(label='हुबहू आवाज़ आउटपुट')
            btn_clone.click(clone_voice, [input_text, sample, speed_slider], output_clone)
            
        with gr.TabItem('⚡ Standard Fast TTS'):
            with gr.Row():
                with gr.Column():
                    t_text = gr.Textbox(label='टेक्स्ट', lines=5)
                    v_drop = gr.Dropdown(choices=['hi-IN-MadhurNeural', 'hi-IN-SwaraNeural'], label='आवाज़', value='hi-IN-MadhurNeural')
                    spd = gr.Slider(-50, 50, 0, label="Speed %")
                    ptc = gr.Slider(-20, 20, 0, label="Pitch")
                    btn_fast = gr.Button('Quick Generate')
                output_fast = gr.Audio(label='फास्ट ऑडियो')
            btn_fast.click(lambda t, v, s, p: asyncio.run(fast_tts(t, v, s, p)), [t_text, v_drop, spd, ptc], output_fast)

demo.launch(share=True)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("✅ app.py अपडेट हो गया है। पुराने सभी टूल्स बरकरार हैं!")
!python app.py